# CAMUS dataset exploration
Inspect echocardiography images and the LV endocardium annotations before training the U-Net.

In [ ]:
from pathlib import Path
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
TRAINING = Path(os.environ.get("CAMUS_TRAINING_DIR", str(ROOT / "data/camus/training")))
assert TRAINING.is_dir(), f"Dataset not found: {TRAINING}"
pairs = []
for folder in sorted(TRAINING.glob("patient*")):
    for mask in sorted(folder.glob("*_gt.mhd")):
        image = mask.with_name(mask.name.replace("_gt.mhd", ".mhd"))
        if image.exists(): pairs.append((image, mask))
print(f"{len(pairs)} annotated frames across {len(set(x.parent.name for x, _ in pairs))} patients")
assert pairs, "No annotated images found"

In [ ]:
def read(path):
    array = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    return array[0] if array.ndim == 3 else array

fig, axes = plt.subplots(min(4, len(pairs)), 3, figsize=(10, 12), squeeze=False)
for ax, (image_path, mask_path) in zip(axes, pairs[:4]):
    image, mask = read(image_path), read(mask_path)
    lv = mask == 1
    ax[0].imshow(image, cmap="gray"); ax[0].set_title(image_path.stem)
    ax[1].imshow(lv, cmap="gray"); ax[1].set_title("LV endocardium mask")
    ax[2].imshow(image, cmap="gray")
    ax[2].contour(lv, levels=[.5], colors="lime")
    ax[2].set_title("Annotation overlay")
    for a in ax: a.axis("off")
plt.tight_layout()
plt.show()